# L335 MVP Task 4: colour-weighted PGD

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/L335_MiniProject"
GSO_DATA = DRIVE_ROOT + "/cache/gso.tgz"
BRANCH = "main"
PGD_COMMIT = "b5dc8be"
import os
os.environ.update(L335_DRIVE=DRIVE_ROOT, GSO_DATA=GSO_DATA, BRANCH=BRANCH, PGD_COMMIT=PGD_COMMIT)
from google.colab import drive
drive.mount("/content/drive", timeout_ms=600000)
!nvidia-smi --query-gpu=name --format=csv,noheader

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tesla T4


In [ ]:
%%bash
set -e
cd /content
test -d PGD || { git clone -q https://github.com/git-guocc/PGD.git && git -C PGD checkout -q $PGD_COMMIT; }
test -x bin/micromamba || curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
export MAMBA_ROOT_PREFIX=/content/mamba
ENV=/content/mamba/envs/pgd
PY=$ENV/bin/python
C=$L335_DRIVE/cache
if test ! -x $PY; then
  if test -f $C/pgd-explicit.txt; then bin/micromamba create -y -n pgd --file $C/pgd-explicit.txt > /content/pgd_env.log 2>&1
  else bin/micromamba create -y -n pgd -c pytorch3d -c pytorch -c nvidia/label/cuda-11.8.0 -c nvidia -c conda-forge python=3.10 pytorch=2.3.1 torchvision=0.18.1 pytorch-cuda=11.8 pytorch3d cuda-nvcc cuda-cudart-dev cuda-libraries-dev gcc_linux-64=11 gxx_linux-64=11 "mkl<2024.1" "numpy<2" > /content/pgd_env.log 2>&1
  fi
fi
$PY -m pip install -q "numpy<2" lightning==2.4.0 pytorch-lightning==2.4.0 iopath==0.1.10 torch-geometric==2.6.1 point-cloud-utils==0.31.0 pandas tensorboard tqdm scipy
$PY -m pip install -q "numpy<2" pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.3.1+cu118.html
if ls $C/wheels/pgd/*.whl > /dev/null 2>&1; then
  $PY -m pip install -q --no-deps $C/wheels/pgd/*.whl
else
  export PATH=$ENV/bin:$PATH CUDA_HOME=$ENV TORCH_CUDA_ARCH_LIST=7.5 MAX_JOBS=2
  export CC=$ENV/bin/x86_64-conda-linux-gnu-gcc CXX=$ENV/bin/x86_64-conda-linux-gnu-g++
  (cd PGD/chamfer3d && $PY setup.py install > /content/pgd_chamfer3d.log 2>&1)
  (cd PGD/pointops && $PY setup.py install > /content/pgd_pointops.log 2>&1)
fi
cd PGD && $PY -c "import torch, chamfer3d, pointops_cuda; from models.pgd import PGDModel; assert torch.cuda.is_available(); print('PGD ok')"

PGD ok


In [ ]:
%%bash
cd /content
if test ! -d L335_MiniProject; then
  TOKEN=$(/usr/bin/python3 -c "from google.colab import userdata; print(userdata.get('GITHUB_TOKEN'))")
  git clone -q --branch $BRANCH https://$TOKEN@github.com/milanagm/L335_MiniProject.git
fi
cd L335_MiniProject
rm -rf external/PGD && ln -s /content/PGD external/PGD
mkdir -p data
test -d data/gso || tar xzf "$GSO_DATA" -C data
echo "GSO objects: train $(ls data/gso/train | wc -l), val $(ls data/gso/val | wc -l), test $(ls data/gso/test | wc -l)"

GSO objects: train 42, val 6, test 12


In [ ]:
%%bash
cd /content/L335_MiniProject
PYTHONPATH=. /content/mamba/envs/pgd/bin/python tests/gpu/check_cuda_equivalence.py --out results/task4/checks/cuda_equivalence.json

{
  "object": "Cootie_Game_tDhURNbfU5J",
  "device": "cuda",
  "max_abs_diff_to_pgd_patch_based_denoise": {
    "get_frozen_pgd": 9.144842624664307e-05,
    "geometry_only": 9.144842624664307e-05,
    "grayscale": 9.144842624664307e-05,
    "rgb_fusion": 9.144842624664307e-05,
    "gated": 9.144842624664307e-05
  },
  "max_abs_diff_patches_per_pass_20": 5.960464477539062e-07
}


In [ ]:
%%bash
cd /content/L335_MiniProject && mkdir -p results/task4/runs
PY=/content/mamba/envs/pgd/bin/python
CONFIG=configs/task4/colour_weighted_pgd.yaml
RUNS=results/task4/runs
train() {
  name=$1; shift
  if test -f $RUNS/$name/summary.json; then echo "done already: $name"; return; fi
  if test -d $RUNS/$name; then mv $RUNS/$name $RUNS/${name}_incomplete_$(date +%H%M%S); fi
  echo "training $name"
  PYTHONPATH=. $PY -m src.training.train --config $CONFIG --run-name $name "$@" >> results/task4/training.log 2>&1
}
for lr in 1e-05 0.0001 0.001; do
  train lr${lr}_geometry_only_seed2025 --variant geometry_only --seed 2025 --lr $lr
done
LR=$($PY - <<'PY'
import json
runs = "results/task4/runs"
losses = {lr: json.load(open(f"{runs}/lr{lr}_geometry_only_seed2025/summary.json"))["best_val_loss"]
          for lr in ("1e-05", "0.0001", "0.001")}
best = min(losses, key=losses.get)
json.dump({"lr": float(best), "val_loss_per_lr": losses}, open("results/task4/lr_choice.json", "w"), indent=2)
print(best)
PY
)
echo "learning rate: $LR"
for seed in 2025 2026 2027; do
  for variant in geometry_only grayscale rgb_fusion gated; do
    train ${variant}_seed$seed --variant $variant --seed $seed --lr $LR
  done
done

training lr1e-05_geometry_only_seed2025
training lr0.0001_geometry_only_seed2025
training lr0.001_geometry_only_seed2025
learning rate: 0.0001
training geometry_only_seed2025
training grayscale_seed2025
training rgb_fusion_seed2025
training gated_seed2025
training geometry_only_seed2026
training grayscale_seed2026
training rgb_fusion_seed2026
training gated_seed2026
training geometry_only_seed2027
training grayscale_seed2027
training rgb_fusion_seed2027
training gated_seed2027


In [ ]:
%%bash
cd /content/L335_MiniProject
mkdir -p $L335_DRIVE/results && cp -r results/task4 $L335_DRIVE/results/